# Dirichlet process mixture model

Use PyMC to implement the stick breaking algorithm for Dirichlet process mixture modeling.

This generalizes the Dirichlet allocation model for $K \longrightarrow \infty$

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import networkx as nx

from utils import check_save_path, plot_corr

# Hardcoded RNG object for this script for reproducibility
RANDOM_SEED = 5295
rng = np.random.default_rng(RANDOM_SEED)


# Hardcoded path, change to apply to new data
data_path = "data/preprocessed/esa_2018.csv"
vars_info_path = "data/preprocessed/esa_grouped_vars_info.csv"
outputs_folder = "outputs"
allow_overwrite = True

In [ ]:
# Load data and information about variables
data = pd.read_csv(data_path, index_col=0, low_memory=False)
vars_info = pd.read_csv(vars_info_path, index_col=0, low_memory=False)

# Define variables
indicator_vars = (
    pd.Series(vars_info.index[vars_info["type"] == "categorical"])  # change which variables are included later
    .sample(n=20, random_state=np.random.RandomState(rng.bit_generator))  # reduce size for now
    .values
)

training_data = (
    data.loc[:, indicator_vars].sample(
        n=1000, axis=0, random_state=np.random.RandomState(rng.bit_generator)
    )  # reduce size for now
)
num_respondents, num_questions = training_data.shape  # N, J
num_choices_list = [val.max() + 1 for var, val in training_data.items()]  # A

# Plot the initial correlation matrix
plot_corr(training_data, var_names=indicator_vars, tick_label_size=9)
plt.show()

## The Dirichlet process mixture from a simpler model

The responses $X_0, X_1, \dotsc, X_{J - 1}$ of a respondant arise as follows from the data generating process ($\alpha$ is a sparsity parameter, question $j$ has $A_j$ answering options).

A finite class case with $K$ latent classes is shown below.

$\displaystyle \Gamma \sim \mathrm{Dirichlet}\Big(\frac{\alpha}{K} \, \mathbf{1}_K\Big) \quad \big(\text{sparse exchangeable Dirichlet class probability prior}\big)$\
$C \mid \Gamma = \gamma \sim \mathrm{Categorical}(\gamma) \quad \big(\mathrm{supp}(C) \in \{0, \dotsc, K - 1\}, C \text{ is class membership}\big)$

$\Pi_j \mid C = c \sim \mathrm{Dirichlet}(\beta \, \mathbf{1}_{A_j}) \quad \forall j \in \{0, \dotsc, J - 1\} \quad \big(\text{answer distribution for question } j \text{ is different in every latent class}\big)$\
$X_j \mid \Pi_j = \pi_j \sim \mathrm{Categorical}(\pi_j) \quad \forall j \in \{0, \dotsc, J - 1\} \quad \big(\text{answer drawn according to } \pi_j \text{ implicitly depends on } c\big)$

This can be rephrased in terms of a random profile $\Pi = \{\Pi_0, \Pi_1, \dotsc, \Pi_{J - 1}\}$ for the respondent, made up of their vectors of answer probabilities $\Pi_j$ to each question $j$. Now, we impose the finite class structure more implicitly, by drawing this profile from a distribution of class profiles with finite support.

$\displaystyle \Gamma \sim \mathrm{Dirichlet}\Big(\frac{\alpha}{K} \, \mathbf{1}_K\Big)$\
$C \mid \Gamma = \gamma \sim \mathrm{Categorical}(\gamma)$\
$\tilde{\Pi}_c \sim {\Large\times}_{j = 0}^{J - 1} \, \mathrm{Dirichlet}(\beta \, \mathbf{1}_{A_j}) \quad \forall c \in \{0, 1, \dotsc, K - 1\} \quad \big(\text{class profile } \tilde{\Pi}_c \text{ is equal to the respondent's profile } \Pi \text{ if } C = c\big)$

$\displaystyle \Pi \mid \tilde{\Pi}_0 = \tilde{\pi}_0, \tilde{\Pi}_1 = \tilde{\pi}_1, \dotsc, \tilde{\Pi}_{K - 1} = \tilde{\pi}_{K - 1} \sim \sum_{c = 0}^{K - 1} \gamma_c \, \delta_{\{\tilde{\pi}_c\}} \quad \big(\text{class choice implicit in the drawing of a profile, } \Pi = \tilde{\Pi}_C \big)$\
$X_j \mid \Pi_j = \pi_j \sim \mathrm{Categorical}(\pi_j) \quad \forall j \in \{0, 1, \dotsc, J - 1\} \quad \big(\pi_j \in \Delta^{A_j - 1}\big)$

Taking the limit as $K \longrightarrow \infty$ we obtain a Dirichlet process mixture. The class assignment is now implicit in the drawing of one of countably infinite profiles. This corresponds to a finite mixture, as above, with a very high number of classes $K >> 0$ and a very small parameter $\alpha / K$ in the exchangeable Dirichlet prior to generate sparsity (few classes of actual importance).

$G_0 = {\Large\times}_{j = 0}^{J - 1} \, \mathrm{Dirichlet}(\beta \, \mathbf{1}_{A_j})$\
$G \sim \mathrm{DP}(G_0, \alpha) \quad \big(G \text{is a random discrete pmf over (a.s.\ countably many) latent profiles}\big)$

$\Pi \mid G = g \sim g \quad \big(g\text{ is a realization of } G \text{ used to draw the respondent's profile } \Pi\big)$\
$X_j \mid \Pi_j = \pi_j \sim \mathrm{Categorical}(\pi_j) \quad \forall j \in \{0, 1, \dotsc, J - 1\} \quad \big(\pi_j \in \Delta^{A_j - 1}\big)$

In [ ]:
# Simulation of Dirichlet process mixture using stick-breaking
# ...